# FLIR Density-Based Clustering — Project Report

## 1. Objetivo

Comparar DBSCAN, OPTICS y HDBSCAN para identificar grupos de contenidos visualmente relacionados, estables y trazables, candidatos a unidades indivisibles de particionamiento. La evaluación usa 1459 contenidos únicos; el mapping a las 1657 ocurrencias permanece disponible. Esta fase termina con **candidatos de clustering**, sin generar train/val/test ni resultados de YOLO.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Image, Markdown, display

report = Path("reports/clustering")
tables = report / "tables"
summary = json.loads((report / "summary.json").read_text(encoding="utf-8"))
metadata = json.loads((report / "report_metadata.json").read_text(encoding="utf-8"))
screening = pd.read_csv(tables / "screening.csv")
runs = pd.read_csv(tables / "all_runs.csv")
shortlist = pd.read_csv(tables / "evaluated_shortlist.csv")
candidates = pd.read_csv(tables / "candidates.csv")
references = pd.read_csv(tables / "references.csv")
agreements = pd.read_parquet(tables / "assignment_comparisons.parquet")
def table(frame):
    display(HTML(frame.to_html(index=False, float_format=lambda v: f"{v:.4f}", na_rep="No aplicable / indefinido", border=0, escape=True)))
def figure(name):
    display(Image(filename=str(report / "figures" / name)))
def details(title, frame):
    display(HTML("<details><summary>" + title + "</summary>" + frame.to_html(index=False, float_format=lambda v: f"{v:.4f}", na_rep="No aplicable / indefinido", border=0, escape=True) + "</details>"))
key = ["encoder", "representation", "algorithm", "clustering_space_id"]
display(Markdown(f"**Evidencia ejecutada:** {summary['total_runs']} runs únicos: {summary['screening_runs']} de screening y {summary['additional_seed_runs']} adicionales de semillas. {summary['shortlist_count']} configuraciones en Fase B; {summary['candidate_count']} candidatos Pareto finales y {summary['review_reference_count']} referencias para inspección. Los artefactos se verificaron antes de construir el reporte."))

## 2. Espacios evaluados

Los originales **DINOv2 L2 (384D)** y **CLIP L2 (512D)** son controles metodológicos. La ruta comprometida evalúa también sus candidatos **t-SNE (perplexity 30)** y **PaCMAP (MN_ratio 1.0)** en 2D. Screening usa semilla 0; las semillas 1/2 se reservan a la shortlist. No se sustituye la ruta reducida por los controles originales.

Para vectores unitarios, ||a−b||² = 2(1−cos(a,b)): Euclidean y cosine conservan el orden salvo redondeo de las fuentes. Los ajustes usan distancias euclidianas, float64 de cálculo y búsqueda brute con un hilo, sin renormalización ni z-score. Los archivos float32 originales permanecen intactos. En 2D, la escala se deriva del propio espacio; su densidad no equivale a la original.

Clases, labels, split histórico, secuencia e índice temporal no entran al fit. El orden de entrada se fija por content_id y los labels se devuelven al content_index original.

In [ ]:
table(runs.groupby(["encoder", "representation", "algorithm"]).size().rename("runs").reset_index())

## 3. DBSCAN protocol

min_samples **5/10/20**, incluyendo self. La k-distance corresponde al vecino distinto número min_samples−1, excluyendo self explícitamente incluso con empates a distancia cero. eps se deriva de cuantiles lineales **80/85/90/95/97/99%** en cada espacio. Los eps exactos redundantes se registran y no se repiten; un eps no positivo se registra como no ejecutable, sin inventar una constante.

Al pasar a otra semilla de reducción se mantiene min_samples + cuantil y se recalcula eps. La figura muestra las diferencias de escala que impiden compartir un eps absoluto.

In [ ]:
figure("01_k_distance_scales.png")
details("Parámetros efectivos de todos los DBSCAN ejecutados", runs.loc[runs.algorithm.eq("dbscan"), key + ["reduction_seed", "hyperparameters", "effective_eps", "fit_seconds"]])

## 4. OPTICS protocol

Grid controlado: min_samples **5/10/20**, xi **0.03/0.05/0.10** y min_cluster_size **10/20/30**. Se usa extracción xi, max_eps infinito y predecessor_correction=True. Reachability, core distances y orden de recorrido se conservan en los artefactos locales; infinito en estos diagnósticos es un marcador válido de inicio/desconexión.

Se conservan todas las ejecuciones solicitadas, incluidos resultados redundantes en estructura o desfavorables. La cantidad de grupos no funciona como un score de calidad.

## 5. HDBSCAN protocol

Implementación de scikit-learn ya disponible en el entorno, sin añadir la biblioteca externa hdbscan. min_cluster_size **10/20/30/50** × min_samples **5/10/20**, principalmente EOM, cluster_selection_epsilon=0, alpha=1, allow_single_cluster=False. Aquí min_samples incluye el propio punto; la convención difiere en uno respecto de la biblioteca externa.

Se guardan membership probabilities cuando están disponibles, con cero para noise. Outlier scores y cluster persistence no son requisitos de esta implementación; su disponibilidad se registra. Las probabilidades de pertenencia no son probabilidades de una clase YOLO. No se cambia de biblioteca para obtener métricas accesorias.

## 6. Screening

Fase A estudia los seis espacios y guarda todas las configuraciones, sus métricas, tiempos y flags. La tabla resumida siguiente incluye también las ejecuciones posteriores de semillas; la tabla desplegable conserva el screening completo, sin omitir resultados desfavorables. Los CSV completos están en la carpeta local `reports/clustering/tables/`.

In [ ]:
table(pd.read_csv(tables / "overview.csv"))
details("Screening completo: incluidos los runs desfavorables", screening[key + ["hyperparameters", "n_clusters_excluding_noise", "noise_fraction", "largest_cluster_fraction", "silhouette_original_space", "weighted_mean_intra_cluster_similarity", "temporal_recall@5", "visual_neighbor_coherence@10", "pareto_stage_a", "shortlist_stage_a", "fit_seconds"]])

## 7. Noise and cluster structure

Noise permanece **−1** y no se convierte en singletons. Se reportan todos los puntos, cobertura agrupada, cuantiles de tamaños y fracción del grupo mayor sobre N total. all_noise y single_cluster (n_clusters≤1) identifican degeneración para la shortlist. nearly_all_noise (≥0.95) y dominant_cluster (≥0.90 del total) son avisos descriptivos; por sí solos no excluyen configuraciones.

Menos ruido o más grupos no implican mejor solución. Las configuraciones de cero/un grupo quedan fuera de selección pero se conservan en los resultados.

In [ ]:
figure("02_screening_structure.png")
flags = runs.groupby(["encoder", "representation", "algorithm"])[["all_noise", "single_cluster", "nearly_all_noise", "dominant_cluster"]].sum().reset_index()
table(flags)
display(Markdown(f"**Observado:** {summary['degenerate_runs']} runs con cero/un grupo; número de grupos entre {summary['clusters_min']} y {summary['clusters_max']}; noise entre {summary['noise_fraction_min']:.4f} y {summary['noise_fraction_max']:.4f}. Los flags se solapan y no deben sumarse como casos distintos."))

## 8. Visual coherence

Silhouette principal se calcula **en el L2 original del encoder**, aun si el clustering se ajustó en 2D. Se excluye noise y se exige 2≤n_clusters<n_clustered; los casos indefinidos no se convierten en cero. La silhouette del espacio de clustering se registra por separado.

La cohesión visual usa coseno original entre pares únicos i<j de cada grupo: media, mediana y Q1/Q3. El agregado principal se pondera por cantidad de pares, y se conserva también la ponderación por miembros. La retención de vecinos @5/10/20 usa los vecinos originales: excluye queries noise, mantiene vecinos noise como no retenidos y reporta cobertura. Es cohesión de la representación, no validación semántica de toda la población.

In [ ]:
figure("03_original_space_cohesion.png")
table(references[["reference", *key, "silhouette_original_space", "silhouette_clustering_space", "weighted_mean_intra_cluster_similarity", "median_cluster_intra_similarity", "visual_neighbor_coherence@5", "visual_neighbor_coherence@10", "visual_neighbor_coherence@20", "visual_query_coverage"]])

## 9. Temporal coherence

Procedencia posterior por consenso de ocurrencias: secuencias conocidas, fracción dominante, entropía Shannon en bits y cobertura de datos conocidos. Los desconocidos no se convierten en una secuencia adicional. La fracción dominante se calcula entre miembros con secuencia conocida y se acompaña de cobertura.

Temporal recall@1/5/10 cuenta pares de contenidos distintos de la misma secuencia válida con |delta|≤k, incluyendo delta=0 cuando exista. El denominador contiene todos los pares elegibles, también si sus extremos son noise; solo se retienen pares dentro del mismo grupo no-noise. Son relaciones de índices inferidos, sin timestamps verificados. La selección con estas métricas es exploratoria y no constituye validación temporal independiente.

In [ ]:
table(references[["reference", "encoder", "representation", "algorithm", "weighted_dominant_sequence_fraction", "weighted_sequence_entropy_bits", "clustered_sequence_coverage", "temporal_recall@1", "temporal_recall@5", "temporal_recall@10", "temporal_pairs@1", "temporal_pairs@5", "temporal_pairs@10"]])

## 10. Stability — ARI / AMI

ARI y AMI comparan pequeñas perturbaciones controladas, no algoritmos diferentes como sustituto de estabilidad. AMI usa normalización aritmética. Se distinguen:

- **All-points:** incluye noise como categoría −1; un gran conjunto compartido de noise puede dominar el acuerdo.
- **Common-clustered:** intersección de contenidos no-noise; se publican N y cobertura. Con N<2 no se define el acuerdo. Una partición de un solo grupo se marca trivial aunque sklearn produzca un valor.

En reducciones se comparan semillas 0/1, 0/2 y 1/2 de la misma configuración conceptual. Original L2 no tiene semilla de reducción; esa estabilidad es no aplicable. La robustez local se evalúa para toda shortlist mediante cuantiles eps vecinos, xi vecino o min_samples/min_cluster_size vecinos. Se mantienen comparaciones contra vecinos degenerados; no se exige estabilidad perfecta.

In [ ]:
figure("04_assignment_stability.png")
agreement_fields = [name for name in shortlist.columns if name.startswith(("parameters_", "seeds_"))]
details("Estabilidad completa de la shortlist: medias, mínimos, cobertura y trivialidad", shortlist[[*key, *agreement_fields]])
table(agreements.groupby(["encoder", "representation", "kind"]).agg(comparisons=("reference_id", "size"), ari_all_mean=("all_points_ari", "mean"), ari_common_mean=("common_clustered_ari", "mean"), ami_all_mean=("all_points_ami", "mean"), ami_common_mean=("common_clustered_ami", "mean"), common_min_coverage=("common_clustered_coverage", "min"), trivial_common=("common_clustered_trivial", "sum")).reset_index())

## 11. Original embeddings vs reduced spaces

Los controles originales y la ruta reducida se evalúan con la misma referencia visual original y políticas de ruido. La tabla muestra las referencias de revisión; los candidatos alternativos permanecen disponibles. Diferencias de cobertura dificultan comparar silhouette o cohesión aisladamente. No se concluye que una reducción mejore el agrupamiento solo porque separe visualmente los colores.

In [ ]:
comparison_fields = ["reference", "encoder", "representation", "algorithm", "n_clusters_excluding_noise", "noise_fraction", "largest_cluster_fraction", "silhouette_original_space", "weighted_mean_intra_cluster_similarity", "weighted_dominant_sequence_fraction", "temporal_recall@5", "visual_neighbor_coherence@10", "parameters_common_clustered_ari_min", "parameters_common_clustered_ami_min", "seeds_common_clustered_ari_min", "seeds_common_clustered_ami_min"]
table(references.reindex(columns=comparison_fields))
for name in metadata["figure_names"]:
    if name.endswith("_clusters.png"):
        figure(name)

## 12. DINOv2 vs CLIP

Se comparan resultados descriptivos bajo el mismo protocolo y población, sin concatenar encoders ni imponer un mismo eps. El rango/calibración del coseno difiere entre sus espacios, por lo que una cohesión numéricamente mayor de CLIP no establece superioridad. Las secuencias disponibles tampoco constituyen ground truth de escenas. No se calculan métricas de detector en esta etapa.

In [ ]:
table(runs.groupby(["encoder", "representation"]).agg(runs=("clustering_space_id", "size"), median_clusters=("n_clusters_excluding_noise", "median"), median_noise=("noise_fraction", "median"), median_silhouette_original=("silhouette_original_space", "median"), median_visual_cohesion=("weighted_mean_intra_cluster_similarity", "median"), median_temporal_retention=("temporal_recall@5", "median"), median_visual_retention=("visual_neighbor_coherence@10", "median")).reset_index())

## 13. Shortlist / Pareto candidates

Fase A: frente Pareto por encoder × representación × algoritmo. Se maximizan silhouette original, cohesión visual, retención visual@10 y temporal@5; se minimizan noise y fracción del cluster mayor. No hay suma de pesos. Hasta tres anclas: extremo de silhouette, retención visual y temporal; duplicados se completan por configuration_id. Se conservan el frente completo y el motivo de selección.

Fase B: Pareto entre la shortlist de cada encoder × representación, añadiendo mínimos ARI/AMI common-clustered de robustez local y, en reducidos, entre semillas. Se requieren comparaciones no triviales y semillas no degeneradas. Si faltan candidatos no se relaja la regla. La referencia de figuras maximiza silhouette original dentro del frente final; es una preferencia de revisión, no un ganador universal. Clases e historial de splits no son criterios.

In [ ]:
details("Shortlist y motivos de revisión", shortlist[[*key, "hyperparameters", "shortlist_reason", "eligible_stage_b", "pareto_stage_b", "reference_for_review"]])
table(screening.groupby(["encoder", "representation", "algorithm"])[["eligible_stage_a", "pareto_stage_a", "shortlist_stage_a"]].sum().reset_index())

## 14. Cluster exemplars

Se eligen grupos pequeño, cercano al tamaño mediano, grande, de mayor fracción dominante de secuencia y de mayor entropía de secuencia. Estas reglas pueden seleccionar el mismo grupo varias veces; “mayor entropía disponible” no garantiza diversidad temporal. Empates por cluster_id. Noise usa una muestra reproducible de hasta cuatro miembros, semilla 0.

Cada fila muestra el **medoide de distancia euclidiana original** y miembros cercanos/intermedios/lejanos respecto de él. No se utiliza el centroide 2D como representante semántico. Las imágenes se leen del ZIP en runtime, se verifica su hash y se representan en memoria; no se extraen ni versionan. Los content IDs de selección permanecen en tablas privadas.

Los ejemplos permiten una revisión visual acotada. No constituyen una auditoría representativa de todos los grupos ni confirman minería ilegal.

In [ ]:
for name in metadata["figure_names"]:
    if name.endswith("_exemplars.png"):
        figure(name)
display(Markdown(f"Se representaron {metadata['exemplar_image_count']} posiciones de imágenes en {metadata['reference_count']} galerías de referencia. Las posiciones pueden repetir contenido cuando distintos roles eligen el mismo grupo."))

## 15. Historical split interpretation

Se conserva la unión de todas las pertenencias históricas de las ocurrencias de cada grupo, incluidas pertenencias múltiples de un único contenido duplicado. Un grupo que cruza train/val/test históricos no es un mal grupo ni recibe penalización en selección. Puede ayudar a explicar por qué debe estudiarse la partición por grupos, pero no determina por sí solo la cantidad de leakage residual.

In [ ]:
table(pd.read_csv(tables / "historical_memberships.csv"))
figure("05_historical_split_interpretation.png")

## 16. Limitations

Las métricas describen geometría de embeddings y procedencia inferida, sin ground truth de escenas. La silhouette excluye noise y cambia de población entre runs: debe interpretarse junto con cobertura. ARI/AMI all-points pueden estar dominados por ruido compartido; common-clustered puede tener menor cobertura y comparaciones triviales. La estabilidad de reducción y la robustez de parámetros son perturbaciones distintas.

La selección usa un protocolo explícito de Pareto y anclas, no una función validada de rendimiento del detector. El grid es acotado y no agota configuraciones; las alternativas y degeneraciones se conservan. No se valida una mejora de DINOv2/CLIP, de reducción o de detección a partir de una figura. Los colores/IDs de cluster son locales a cada ejecución y no tienen correspondencia automática entre figuras.

La temporalidad sigue PARTIAL: no hay timestamps/FPS verificados. Clases YOLO no se usan como pureza de escena ni para elegir candidatos. Los conflictos de anotación y ocurrencias históricas permanecen intactos. Esta fase no realiza balanceo, nuevos splits, entrenamiento ni evaluación YOLO.

## 17. Candidate clustering configurations

La tabla conserva todos los candidatos Pareto finales con su clustering_space_id y parámetros. Las referencias de figuras son solo un subconjunto para inspección. El algoritmo, representación, semilla de reducción, eps efectivo y versiones se registran en metadata local, junto con huellas de entrada/salida. Los labels por contenido y medoides se mantienen fuera de Git.

In [ ]:
table(candidates[[*key, "hyperparameters", "effective_eps", "n_clusters_excluding_noise", "noise_fraction", "silhouette_original_space", "weighted_mean_intra_cluster_similarity", "temporal_recall@5", "visual_neighbor_coherence@10", "reference_for_review"]])
display(Markdown("**Trazabilidad:** los ajustes conservan el commit y worktree reales al ejecutar, con hashes del código disponible. La metadata experimental no se reescribe para atribuir el cálculo a un commit posterior."))

## 18. Next step

Diseñar **cluster-aware splitting** tras revisar los candidatos y decidir una política explícita para noise. Noise todavía no equivale a singletons independientes y no debe repartirse automáticamente. Mantener íntegro cada grupo seleccionado y todos los frame_id asociados a un content_id; medir overlap exacto, similitud visual residual y relaciones temporales entre particiones.

Comparar después con la partición histórica y un baseline aleatorio reproducible, documentando unidad, cobertura de clases y conflictos de anotación. La comparación controlada de Precision/Recall/mAP del detector será posterior. **No se genera ninguna partición nueva en esta entrega.**